# Домашнее задание: Pydantic


## Важно!

- При выполнении задания используем точные типы (`EmailStr`, `HttpUrl`, `SecretStr`, `Decimal`, конкретные `Enum`).
- Придерживаемся принципа разделения валидаций: проверка поля — в `field_validator`, сквозные зависимости — в `model_validator`


## Задача 1. Профиль пользователя (валидация полей)

Постройте модель профиля пользователя для внутренней CRM:

**Требования**
1. Обязательные поля: `id: UUID`, `email: EmailStr`, `name: str`.
2. Опциональные поля: `website: HttpUrl | None`, `bio: str | None`.
3. Пароль хранится как `SecretStr`, должен быть не короче 8 символов.
4. Имя (`name`) нормализуйте: тримминг + одна пробельная последовательность между словами + первая буква каждого слова заглавная.
5. Если указан `website`, домен сайта не должен совпадать с доменом `email` (смысл: личный сайт != корпоративная почта).

Подсказки: используйте `field_validator` для нормализации и локальных проверок; и `model_validator(mode="after")` для проверки зависимости `email` ↔ `website`.


In [3]:
!pip install -U pydantic[email,timezone] -q

zsh:1: no matches found: pydantic[email,timezone]


In [5]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID

class UserProfile(BaseModel):
    # опишите поля согласно требованиям
    id: UUID
    email: EmailStr
    name: str
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None
    password: SecretStr

    # нормализация имени
    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        # Убираем лишние пробелы и нормализуем интервалы
        parts = v.strip().split()
        normalized = " ".join(word.capitalize() for word in parts)
        return normalized

    # проверка длины пароля
    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        real = v.get_secret_value()
        if len(real) < 8:
            raise ValueError("Пароль меньше 8 символов")
        return v

    # сквозная проверка доменов email/website
    @model_validator(mode="after")
    def check_domains(self):
        if self.website:
            email_domain = self.email.split("@")[-1].lower()
            website_domain = self.website.host.lower()

            if email_domain == website_domain:
                raise ValueError(
                    "Email домена должен отличаться от Email website"
                )
        return self


In [6]:
from uuid import uuid4

u = UserProfile(
    id=uuid4(),
    email="JohnBolt@gmail.com",
    name="   john   bold  ",
    website="https://johnbolt.org",
    password=SecretStr("superpass")
)

u

UserProfile(id=UUID('86b3b0f9-f06c-4f63-b282-16f85cc779b1'), email='JohnBolt@gmail.com', name='John Bold', website=HttpUrl('https://johnbolt.org/'), bio=None, password=SecretStr('**********'))

In [8]:
UserProfile(
    id=uuid4(),
    email="john@test.com",
    name="john",
    website="https://test.com",
    password=SecretStr("superpass")
)

ValidationError: 1 validation error for UserProfile
  Value error, Email домена должен отличаться от Email website [type=value_error, input_value={'id': UUID('20e87d47-a14...SecretStr('**********')}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

## Задача 2. Валидация функции заказа (`@validate_call`)

Реализуйте функцию `place_order`, которая принимает:
- `user_id: UUID`
- `sku: str` (артикул, только заглавные буквы/цифры, длина 3–12)
- `quantity: int` (>0)
- `price: Decimal` (>= 0), округляется банковским методом до 2 знаков

Функция должна возвращать словарь с ключами: `user_id`, `sku`, `quantity`, `price`, `amount` (quantity × price).

Используйте `@validate_call` и локальные проверки через обычный код (или вспомогательные валидаторы `TypeAdapter` не используем).


In [9]:
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
import re

# реализуйте функцию с @validate_call
@validate_call
def place_order(
    user_id: UUID,
    sku: str,
    quantity: int,
    price: Decimal
):
    # ---- Проверка SKU ----
    # Только заглавные буквы и цифры, длина 3–12
    if not re.fullmatch(r"[A-Z0-9]{3,12}", sku):
        raise ValueError("SKU должен содержать A-Z и 0-9, длина 3–12")

    # ---- Проверка количества ----
    if quantity <= 0:
        raise ValueError("Количество должны быть > 0")

    # ---- Проверка цены ----
    if price < 0:
        raise ValueError("Цена должна быть >= 0")

    # Банковское округление до 2 знаков
    price = price.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    # ---- Расчёт суммы ----
    amount = (price * quantity).quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    return {
        "user_id": user_id,
        "sku": sku,
        "quantity": quantity,
        "price": price,
        "amount": amount
    }


In [10]:
from uuid import uuid4
from decimal import Decimal

place_order(
    user_id=uuid4(),
    sku="ABC123",
    quantity=3,
    price=Decimal("10.005")
)

{'user_id': UUID('ad271f81-de00-481c-a47a-b710e0264f63'),
 'sku': 'ABC123',
 'quantity': 3,
 'price': Decimal('10.00'),
 'amount': Decimal('30.00')}

## Задача 3. Модель заказа с бизнес-правилами

Смоделируйте заказ в магазине цифровых товаров.

**Требования**
- `OrderStatus: Enum` со значениями `new`, `paid`, `delivered`, `canceled`.
- Модель `OrderItem`:
  - `sku: str` как в задаче 2
  - `qty: int` (>0)
  - `unit_price: Decimal` (>=0) округление до 2 знаков
- Модель `Order`:
  - `id: UUID`
  - `user_email: EmailStr`
  - `items: list[OrderItem]` (не пустой)
  - `status: OrderStatus = 'new'`
  - `created_at: datetime` (по умолчанию `datetime.utcnow`)
  - Расчитанное поле `total: Decimal` — сумма по всем позициям
  - В `model_validator(mode="after")` запретите переход в `paid`/`delivered` при `total == 0` и запретите пустые корзины.

**Важно:** используйте только инструменты `pydantic` и стандартную библиотеку.


In [12]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from datetime import datetime, UTC
from enum import Enum

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")

class OrderStatus(str, Enum):
    # перечислите статусы
    new = "new"
    paid = "paid"
    delivered = "delivered"
    canceled = "canceled"

class OrderItem(BaseModel):
    # опишите поля
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        if not SKU_RE.fullmatch(v):
            raise ValueError("SKU должно быть [A-Z0-9]{3,12}")
        return v

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError("qty должно быть > 0")
        return v

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, v: Decimal) -> Decimal:
        if v < 0:
            raise ValueError("unit_price должен быть >= 0")
        return v.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

class Order(BaseModel):
    # опишите поля
    id: UUID
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.new
    created_at: datetime = datetime.now(UTC)

    @property
    def total(self) -> Decimal:
        total = Decimal("0.00")
        for item in self.items:
            total += (item.unit_price * item.qty)
        return total.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    @model_validator(mode="after")
    def check_business_rules(self):
        # Корзина не должна быть пустой
        if not self.items:
            raise ValueError("Корзина не должна быть пустой")

        # Общая сумма == 0, но статус изменён на paid/delivered
        if self.total == 0 and self.status in {OrderStatus.paid, OrderStatus.delivered}:
            raise ValueError(
                "Общая сумма == 0"
            )
        return self


In [13]:
from uuid import uuid4
from decimal import Decimal

order = Order(
    id=uuid4(),
    user_email="user@example.com",
    items=[
        OrderItem(sku="ABC123", qty=2, unit_price=Decimal("10.005")),
        OrderItem(sku="XYZ999", qty=1, unit_price=Decimal("5.00")),
    ],
)

order.total

Decimal('25.00')

In [14]:
Order(
    id=uuid4(),
    user_email="u@e.com",
    items=[OrderItem(sku="AAA111", qty=1, unit_price=Decimal("0"))],
    status=OrderStatus.paid
)

ValidationError: 1 validation error for Order
  Value error, Общая сумма == 0 [type=value_error, input_value={'id': UUID('f648763b-4d7...derStatus.paid: 'paid'>}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

## Задача 4. Конфигурация приложения (`BaseSettings`)

Опишите настройки подключения к внешнему API:

- `APISettings(BaseSettings)` с полями:
  - `base_url: HttpUrl`
  - `token: SecretStr`
  - `timeout_sec: int = 5` (1–60)
  - `retries: int = 2` (0–10)
- Используйте `model_config = ConfigDict(env_prefix="API_", env_file=".env", extra="ignore")`
- Проверьте, что значения корректно читаются из переменных окружения.

В тесте ниже среда заполняется вручную.


In [15]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator

class APISettings(BaseSettings):
    # поля и валидации
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5
    retries: int = 2

    # пример проверки диапазона для timeout_sec / retries
    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info):
        if info.field_name == "timeout_sec":
            if not (1 <= v <= 60):
                raise ValueError("timeout_sec должно быть между 1 и 60")
        elif info.field_name == "retries":
            if not (0 <= v <= 10):
                raise ValueError("retries должно быть между 0 и 10")
        return v

    model_config = ConfigDict(
        # настройте env_prefix и прочие опции
        env_prefix="API_",
        env_file=".env",
        extra="ignore"
    )


In [16]:
os.environ["API_BASE_URL"] = "https://example.com/api"
os.environ["API_TOKEN"] = "supersecret"
os.environ["API_TIMEOUT_SEC"] = "12"
os.environ["API_RETRIES"] = "3"

s = APISettings()
s

APISettings(base_url=HttpUrl('https://example.com/api'), token=SecretStr('**********'), timeout_sec=12, retries=3)

In [17]:
os.environ["API_TIMEOUT_SEC"] = "100"
APISettings()

ValidationError: 1 validation error for APISettings
timeout_sec
  Value error, timeout_sec должно быть между 1 и 60 [type=value_error, input_value='100', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

## Задача 5. Извлечение из ORM (`from_attributes=True`)

Создайте простую SQLAlchemy-модель `SAUser(id, email, is_active)` (in-memory, без БД) и соответствующую модель Pydantic:

- Pydantic-модель `UserOut` с полями `id: UUID`, `email: EmailStr`, `is_active: bool`.
- Включите поддержку `from_attributes` в `model_config`.
- Создайте инстанс `SAUser` и провалидируйте его через `UserOut.model_validate(sa_user_instance)`.

Проверьте, что преобразование сработало.


In [18]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4
from pydantic import BaseModel, EmailStr, ConfigDict

Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active

class UserOut(BaseModel):
    # опишите поля и включите from_attributes
    id: UUID
    email: EmailStr
    is_active: bool
    model_config = ConfigDict(
        # включите режим атрибутов
        from_attributes=True
    )


In [19]:
sa_user = SAUser(email="test@example.com", is_active=False)

out = UserOut.model_validate(sa_user)
out

UserOut(id=UUID('7be0d691-cfc5-4351-ac17-c7814e05b553'), email='test@example.com', is_active=False)

In [20]:
type(out.id)

uuid.UUID

## Задача 6. JSON Schema и дружелюбные ошибки

1. Для модели из задачи 3 сгенерируйте JSON Schema (метод `model_json_schema`) и запишите его в переменную `ORDER_SCHEMA`.
2. Реализуйте функцию `safe_create_order(data: dict) -> tuple[bool, str]`, которая:
   - пытается создать `Order` из входного `dict`,
   - при успехе возвращает `(True, "<total=...>")`,
   - при ошибке возвращает `(False, "<короткое сообщение об ошибке>")` без стек-трейса.

Не используйте сторонние библиотеки.


In [21]:
# Используем модели из задачи 3: OrderStatus, OrderItem, Order

ORDER_SCHEMA = Order.model_json_schema()  # сгенерируйте схему

def safe_create_order(data: dict) -> tuple[bool, str]:
    # реализуйте безопасное создание заказа
    try:
        order = Order(**data)
        return True, f"<total={order.total}>"
    except Exception as e:
        # Короткое сообщение без стектрейса
        return False, str(e)


In [25]:
from uuid import uuid4
from decimal import Decimal

# Успешный заказ
data1 = {
    "id": str(uuid4()),
    "user_email": "user@example.com",
    "items": [
        {"sku": "ABC123", "qty": 2, "unit_price": Decimal("10.00")},
        {"sku": "XYZ999", "qty": 1, "unit_price": Decimal("5.00")},
    ],
}

safe_create_order(data1)

# # Ошибочный заказ (пустая корзина)
# data2 = {
#     "id": str(uuid4()),
#     "user_email": "user@example.com",
#     "items": [],
# }
#
# safe_create_order(data2)

(True, '<total=25.00>')